# Singularity OCI Installation

This notebook contains the operations needed to install the Singularity OCI runtime on a Kubernetes cluster, \
alongside containerd, setting up the runtime class and a pod to test the installation.

Running this notebook was facilitated by running the following commands on the remote node and then connecting the \
Jupyter notebook to the remote node:

```bash
python3 -m pip install bash_kernel notebook && \
python3 -m bash_kernel.install &&  \
python3 -m notebook --kernel bash --no-browser --port=8080 --ip=*
```

In [2]:
# Verifying connection
hostname

node19.lab.example


## Removing the Node from the Cluster

Before we start, we should make sure that we dont interfere with the cluster. \
For this reason we will remove the node from the cluster, and stop the kubelet service.

```bash
for i in {17..19}; do
    kubectl cordon heydar$proxy.example
    kubectl drain heydar$proxy.example --ignore-daemonsets --delete-emptydir-data
    kubectl taint nodes heydar$proxy.example key=value:NoSchedule
    ssh heydar$i  "sudo systemctl disable kubelet && sudo systemctl stop kubelet"
    ssh heydar$i "sudo systemctl disable containerd && sudo systemctl stop containerd"
done
```


## Installing pre-requisites

### The first pre-requisite is to have singularity installed on the remote node

In [4]:
# Installing dependencies
sudo apt-get update > /dev/null
sudo apt-get install -y \
    git \
    socat \
    inotify-tools \
    libseccomp-dev \
    autoconf \
    automake \
    cryptsetup \
    fuse2fs \
    git \
    fuse \
    libfuse-dev \
    libglib2.0-dev \
    libseccomp-dev \
    libtool \
    pkg-config \
    runc \
    squashfs-tools \
    squashfs-tools-ng \
    uidmap \
    wget \
    zlib1g-dev \
    > /dev/null

# Installing go 
if [ ! $(which go) ]; then
    wget https://go.dev/dl/go1.22.2.linux-amd64.tar.gz
    sudo rm -rf /usr/local/go && sudo tar -C /usr/local -xzf go1.22.2.linux-amd64.tar.gz
fi

# Installing golangci-lint
if [ ! -f $(go env GOPATH)/bin/golangci-lint ]; then
    echo "Installing golangci-lint"
    curl -sSfL https://raw.githubusercontent.com/golangci/golangci-lint/master/install.sh | sh -s -- -b $(go env GOPATH)/bin
    echo 'export PATH=$PATH:$(go env GOPATH)/bin' >> ~/.bashrc
    source ~/.bashrc
fi

# And verifying go installation
export PATH=$PATH:/usr/local/go/bin
go version


E: The repository 'https://apt.kubernetes.io kubernetes-xenial Release' no longer has a Release file.
go version go1.18.1 linux/amd64


## Installing Singularity

With the pre-requisites installed, we can now install Singularity on the remote node.



In [10]:
if [ ! -d ~/singularity-repo ]; then
   git clone --recurse-submodules https://github.com/sylabs/singularity.git ~/singularity-repo && \
        cd singularity-repo && \
        git checkout --recurse-submodules v4.1.2 && \
fi

cd ~/singularity-repo
./mconfig
make -C builddir
sudo make -C builddir install

bash: syntax error near unexpected token `fi'
Configuring for project `singularity-ce' with languages: C, Golang
=> running pre-basechecks project specific checks ...
=> running base system checks ...
 checking: host C compiler... cc
 checking: host C++ compiler... c++
 checking: host Go compiler (at least version 1.20)... /usr/local/go/bin/go
 checking: host C compiler option -Wall... yes
 checking: host C compiler option -Werror... yes
 checking: host C compiler option -Wfatal-errors... yes
 checking: host C compiler option -Wno-unknown-warning-option... yes
 checking: host C compiler option -Wstrict-prototypes... yes
 checking: host C compiler option -Wpointer-arith... yes
 checking: host C compiler option -Wbad-function-cast... yes
 checking: host C compiler option -Woverlength-strings... yes
 checking: host C compiler option -Wframe-larger-than=2047... yes
 checking: host C compiler option -Wno-sign-compare... yes
 checking: host C compiler option -Wclobbered... yes
 checking: hos

## Installing Singularity OCI

Now that we have Singularity installed, we can install the Singularity OCI runtime.



In [12]:
if [ ! -d /usr/local/bin/sycri ]; then
  git clone https://github.com/sylabs/singularity-cri.git && \
    cd singularity-cri && \
    git checkout tags/v1.0.0-beta.5 -b v1.0.0-beta.5 && \
    make && \
    sudo make install
    rm -rf singularity-cri
fi

# Verifying installation

sycri --help


Cloning into 'singularity-cri'...
remote: Enumerating objects: 7527, done.
remote: Counting objects: 100% (4/4), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 7527 (delta 0), reused 2 (delta 0), pack-reused 7523
Receiving objects: 100% (7527/7527), 6.94 MiB | 5.52 MiB/s, done.
Resolving deltas: 100% (3751/3751), done.
Updating files: 100% (1836/1836), done.
Updating files: 100% (296/296), done.
Switched to a new branch 'v1.0.0-beta.5'
 GO bin/sycri
 INSTALL /usr/local/bin/sycri
 INSTALL /usr/local/etc/sycri/sycri.yaml
Usage of sycri:
  -alsologtostderr
    	log to standard error as well as files
  -config string
    	path to config file (default "/usr/local/etc/sycri/sycri.yaml")
  -log_backtrace_at value
    	when logging hits line file:N, emit a stack trace
  -log_dir string
    	If non-empty, write log files in this directory
  -logtostderr
    	log to standard error instead of files
  -stderrthreshold value
    	logs at or above this threshold go to stderr
  -v

## Setting up a Systemd Service for Singularity OCI

Now that we do have the binary, we want to make sure that the daemon is always running. \
We can do this by creating a systemd service for the Singularity OCI runtime.



In [13]:
# Creating a systemd service
cat <<EOF | sudo tee /etc/systemd/system/sycri.service
    [Unit]
    Description=Singularity-CRI
    After=network.target
    StartLimitIntervalSec=0

    [Service]
    Type=simple
    Restart=always
    RestartSec=1
    ExecStart=/usr/local/bin/sycri
    Environment="PATH=/usr/local/libexec/singularity/bin:/bin:/sbin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin"

    [Install]
    WantedBy=multi-user.target

EOF

# And enabling the service

sudo systemctl enable sycri && \
  sudo systemctl start sycri

# Verifying the service
sudo systemctl status 'sycri'

    [Unit]
    Description=Singularity-CRI
    After=network.target
    StartLimitIntervalSec=0

    [Service]
    Type=simple
    Restart=always
    RestartSec=1
    ExecStart=/usr/local/bin/sycri
    Environment="PATH=/usr/local/libexec/singularity/bin:/bin:/sbin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin"

    [Install]
    WantedBy=multi-user.target

● sycri.service - Singularity-CRI
     Loaded: loaded (]8;;file://node19.lab.example/etc/systemd/system/sycri.service/etc/systemd/system/sycri.service]8;;; enabled; vendor preset: enabled)
     Active: active (running) since Mon 2024-04-22 17:06:37 PDT; 20min ago
   Main PID: 208182 (sycri)
      Tasks: 9 (limit: 309218)
     Memory: 4.3M
        CPU: 143ms
     CGroup: /system.slice/sycri.service
             └─208182 /usr/local/bin/sycri

Apr 22 17:06:37 node19.lab.example systemd[1]: Started Singularity-CRI.
Apr 22 17:06:37 node19.lab.example sycri[208182]: I0422 17:06:37.90082…ck
Apr 22 17:06:37 node19.lab.example sycri[

### Testing the sycri installation

Now that we have a node with the Singularity OCI runtime installed, \
lets test wether and how it works before we add it to the cluster.

This is also done because the sycri has not been maintained for a while, \
 and we want to make sure that it works as expected.


In [15]:
sudo apt install -y cri-tools > /dev/null

# Verifying the installation
crictl --version



crictl version v1.26.0


## Setting up the Runtime Class

Now the Singluarity OCI runtime is installed and running, \
the documentation of singularity wants us to modify the kublelet \
so that all pods will be running with the singularity runtime.

This is what it would want us to do:

```bash
cat > /etc/default/kubelet <<EOF
  KUBELET_EXTRA_ARGS=--container-runtime=remote \
  --container-runtime-endpoint=unix:///var/run/singularity.sock \
  --image-service-endpoint=unix:///var/run/singularity.sock
  EOF

sudo systemctl restart kubelet
```

But instead we want to set up a runtime class for singularity, \
so that the user can choose which runtime to use, but 
the default will be containerd.

But as Kubernetes is not able to connect with CRIs which do not 
support the v1 api, and singularity does only support the v1beta2 api,
this is unfortunately a dead end
